# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
print("All record sets in this dataset:")
for recordset in dataset.record_sets():
    print(f"- @id: {recordset['@id']} | name: {recordset['name'] if 'name' in recordset else 'N/A'}")
    print("  Fields/columns:")
    for field in recordset['field']:
        print(f"    - @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified in the overview.

**Note:** We'll demonstrate with the first available record set as an example.

In [ ]:
record_sets_info = list(dataset.record_sets())
record_set_ids = [rs['@id'] for rs in record_sets_info]
print(f"Available record set @id's: {record_set_ids}")

# We'll use the first record set for demonstration
first_record_set_id = record_set_ids[0]
dataframes = dict()

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for {record_set_id}")
    print("Columns:", df.columns.tolist())
    print()
# Example DataFrame columns and preview for the first record set
print(f"Columns in '{first_record_set_id}':", dataframes[first_record_set_id].columns.tolist())
dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** Update `<numeric_field_id>` and `<group_field_id>` below with appropriate field `@id`s from the record set/field overview above.

In [ ]:
# Choose the record set to analyze
active_record_set_id = first_record_set_id
df = dataframes[active_record_set_id]
# List all columns as a user reference
print("Columns available in the DataFrame:", df.columns.tolist())

# Select a numeric field for analysis (replace with the correct @id from overview if needed)
# For demonstration, we'll try to auto-detect a numeric field
numeric_candidates = df.select_dtypes(include='number').columns
if len(numeric_candidates) == 0:
    print("No numeric columns found in the record set.")
else:
    numeric_field = numeric_candidates[0]
    print(f"Using '{numeric_field}' as the numeric field.")
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, normalized_col]].head())

    # Choose a group field (non-numeric, if exists)
    group_candidates = [c for c in df.columns if c != numeric_field and df[c].dtype == object]
    if len(group_candidates) > 0:
        group_field = group_candidates[0]
        print(f"Grouping by '{group_field}'...")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found in the columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_candidates) > 0:
    # Distribution of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouping available, visualize group means
    if len(group_candidates) > 0:
        plt.figure(figsize=(10,5))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        group_means.plot(kind='barh')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.show()


## 6. Conclusion
In this notebook, we demonstrated exploring the FAIR^2 mass spectrometry dataset using `mlcroissant`.
- We loaded dataset metadata and listed available record sets and field `@id`s for reference.
- We extracted records from record sets, performed basic filtering, normalization, and grouping using `@id`-based columns.
- We visualized numeric distributions and group summaries.

You can adapt this notebook by changing the field or record set `@id` variables to suit your specific analysis needs.